# Loading a freeing-vision-benchmark session

This notebook loads the raw data of one session recorded with `src/main-debug-spinnaker.bonsai` into pandas data
frames, following the procedure of the open-rigs
[`load_dataset.ipynb`](https://github.com/open-rigs/open-rigs/blob/aeon-api/examples/data_loading/load_dataset.ipynb)
example: streams are declared with `swc.aeon.schema.streams.Device` and read with `swc.aeon.io.api.load`, using the
open-rigs readers for Harp registers. The readers and loaders specific to this workflow live in `session_io.py`
next to this notebook.

A session directory contains:

- `BehaviorBoard/`: one Harp binary file per logged register (`BehaviorBoard_<Address>_<DateTime>.bin`), plus a
  catch-all file `BehaviorBoard_<DateTime>.bin` holding the register dump taken when the board was opened.
- `SubjectCamera/`, `TrackCamera/`: the video (`<Device>.avi`) and one Harp timestamp per frame (`<Device>.csv`).
- `RigSchema/`, `SessionSchema/`: the rig configuration and session metadata as JSON.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import swc.aeon.io.api as aeon
from swc.aeon.io.video import frames

from open_rigs.core.experiment import ExperimentSession
from ucl_open_freeing_vision_benchmark.rig import UclOpenFreeingVisionBenchmarkRig

from session_io import declare_experiment, find_device_schema, frame_count_check, load_camera

## Dataset location

`root` is a single session directory. It can also be any parent directory (for example `sub-Plimbo`) to stitch
together data from several sessions.

Harp devices need the device schema to decode registers. The logging specification requires it to be saved as
`<Device>/<Device>.yml`. If it is not present, `find_device_schema` downloads the schema for the Harp Behavior
board from [harp-tech/device.behavior](https://github.com/harp-tech/device.behavior).

In [ ]:
root = Path("../temp_data/sub-Plimbo/ses-001_date-2026-10-15T15-18-01")
device_yml = find_device_schema(root)

pd.Series(
    {path.relative_to(root).as_posix(): path.stat().st_size for path in sorted(root.rglob("*")) if path.is_file()},
    name="size (bytes)",
).to_frame()

## Session metadata

The rig configuration and session metadata are validated with the pydantic models that generated them, so the
values used by the workflow (camera serial numbers, trigger frequencies, screen settings, ...) are available
as typed objects.

In [ ]:
session = ExperimentSession.model_validate_json(next(root.glob("SessionSchema/*.json")).read_text())
rig = UclOpenFreeingVisionBenchmarkRig.model_validate_json(next(root.glob("RigSchema/*.json")).read_text())

print(session)
print()
print("behavior board:", rig.behavior_board.port_name, rig.behavior_board.camera_trigger_controller)
pd.DataFrame({name: camera.model_dump() for name, camera in [("SubjectCamera", rig.subject_camera), ("TrackCamera", rig.track_camera)]})

## Declaring the experiment

`declare_experiment` maps each device directory to the readers for its data streams. The Behavior board is a
multi-channel Harp device, so `harp_device` creates one reader per register in the schema, accessed as
`experiment.BehaviorBoard.<Register>`. The two cameras are declared with `camera_video`, a reader for the fixed
`<Device>/<Device>.csv` + `<Device>.avi` pair written by this workflow (see `session_io.py`); it produces the same
columns as `open_rigs.io.reader.RawVideo`, so the data frames work with the aeon video helpers.

In [ ]:
experiment = declare_experiment(device_yml)
board = experiment.BehaviorBoard

print(list(experiment.keys()))

## Loading Harp data

`aeon.load` finds every chunk file for a stream, reads it, and stitches all chunks into a single data frame
indexed by time. Harp timestamps (seconds since 1904-01-01) are converted to UTC datetimes. The workflow logs
`AnalogData`, `StartCameras`, `Camera0Frame`, `Camera0Frequency`, `Camera1Frame` and `Camera1Frequency`.

In [ ]:
analog_data = aeon.load(root, board.AnalogData)
analog_data.head()

In [ ]:
start_cameras = aeon.load(root, board.StartCameras)
start_cameras

In [ ]:
camera0_frames = aeon.load(root, board.Camera0Frame)
camera0_frames.head()

## Loading camera frame metadata

The camera data frames hold one row per logged frame with the video file path and frame index, so they work with
the aeon video helpers.

In [ ]:
subject_video = load_camera(root, experiment.SubjectCamera)
track_video = load_camera(root, experiment.TrackCamera)
subject_video.head()

## Frames versus trigger events

Each camera is hardware-triggered by the board (`Camera0` drives the subject camera, `Camera1` the track camera)
and every trigger pulse is logged as a `Camera0Frame` / `Camera1Frame` event. `frame_count_check` compares the
number of frames logged after `StartCameras` with the number of trigger events. Frames before `StartCameras` are
not trigger-driven (the register-dump read reply logged at startup) and are excluded; triggers issued after the
last logged frame are expected, since the workflow stops before their frame arrives, and are reported separately.
A camera passes when the two counts are equal.

In [ ]:
cameras_started = aeon.load(root, board.StartCameras).index.min()

report = pd.DataFrame(
    {
        "SubjectCamera": frame_count_check(subject_video, aeon.load(root, board.Camera0Frame), cameras_started),
        "TrackCamera": frame_count_check(track_video, aeon.load(root, board.Camera1Frame), cameras_started),
    }
)
report

## Selecting a time window

Pass `start` and `end` to load only the data in a time range. Only the chunk files overlapping the window are read.
Here the window is expressed in seconds relative to the first analog sample; use `aeon.to_datetime` and
`aeon.to_seconds` to convert between datetimes and Harp seconds.

In [ ]:
session_start = aeon.to_seconds(analog_data.index[0])
start = aeon.to_datetime(session_start + 5.0)
end = aeon.to_datetime(session_start + 7.0)
window = aeon.load(root, board.AnalogData, start=start, end=end)
window.index.min(), window.index.max(), len(window)

In [ ]:
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(10, 5), height_ratios=[3, 1])
window.plot(ax=axes[0], title="AnalogData")
axes[1].eventplot(
    [
        aeon.load(root, board.Camera0Frame, start=start, end=end).index,
        aeon.load(root, board.Camera1Frame, start=start, end=end).index,
    ],
    lineoffsets=[1, 0],
    colors=["C0", "C1"],
)
axes[1].set_yticks([1, 0], ["Camera0Frame", "Camera1Frame"])
axes[1].set_title("camera trigger events")
plt.tight_layout()

## Reading video frames

The camera data frames include the video file path and frame index for each frame, so they work with the aeon
video helpers. Here the frame closest to the first, middle and last analog sample in the window is read from disk
for both cameras.

In [ ]:
times = window.index[[0, len(window) // 2, -1]]

fig, axes = plt.subplots(2, len(times), figsize=(12, 6))
for row, (name, video) in zip(axes, [("SubjectCamera", subject_video), ("TrackCamera", track_video)]):
    selected = video.iloc[video.index.get_indexer(times, method="nearest")]
    for ax, image, time in zip(row, frames(selected), selected.index):
        ax.imshow(image[:, :, ::-1])
        ax.set_title(f"{name} {time.time()}")
        ax.axis("off")
plt.tight_layout()

## Chunk sizes

aeon selects chunk files for a time window assuming one-hour chunks. Datasets logged with sub-hour chunks load
correctly. If a dataset was logged with `ChunkSize` greater than one hour, set the chunk duration before loading
a time window, otherwise data from a chunk starting before `start` may be missed:

```python
aeon.CHUNK_DURATION = 2  # hours
```